## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-28-13-46-39 +0000,nypost,"US overdose deaths drop to 6-year low, but one...",https://nypost.com/2026/11/28/us-news/us-overd...
1,2026-11-28-13-36-36 +0000,bbc,Argentina's bid to stop Falklands oil drilling...,https://www.bbc.co.uk/news/articles/c84gnq3qq9...
2,2026-11-28-13-32-40 +0000,cbc,Israeli strike kills 2 staff outside Gaza medi...,https://www.cbc.ca/news/world/israeli-strike-k...
3,2026-11-28-13-25-26 +0000,nyt,Ukraine’s Deep Strikes Grow in Scale and Scope...,https://www.nytimes.com/2026/11/28/world/europ...
4,2026-11-28-13-11-17 +0000,nypost,Young endangered right whale dodges hundreds o...,https://nypost.com/2026/11/28/us-news/young-en...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2772/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
351,new,19
476,trump,15
315,thanksgiving,13
187,up,10
304,court,9


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
170,2026-11-27-20-26-11 +0000,wapo,Trump officials to speed president’s projects ...,https://www.washingtonpost.com/politics/2026/1...,53
178,2026-11-27-20-00-00 +0000,missionlocal,"Tenderloin Buzz: Three new businesses, just in...",https://missionlocal.org/2026/11/tenderloin-bu...,52
222,2026-11-27-17-16-23 +0000,nypost,NBC faces backlash for airing anti-Israel ad d...,https://nypost.com/2026/11/27/media/nbc-faces-...,47
128,2026-11-27-23-04-41 +0000,chitri,Shoppers turn out for Black Friday deals as ho...,https://www.chicagotribune.com/2026/11/27/blac...,42
76,2026-11-28-07-08-12 +0000,nypost,Mexican President Claudia Sheinbaum protests n...,https://nypost.com/2026/11/28/world-news/mexic...,42


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
170,53,2026-11-27-20-26-11 +0000,wapo,Trump officials to speed president’s projects ...,https://www.washingtonpost.com/politics/2026/1...
222,44,2026-11-27-17-16-23 +0000,nypost,NBC faces backlash for airing anti-Israel ad d...,https://nypost.com/2026/11/27/media/nbc-faces-...
128,42,2026-11-27-23-04-41 +0000,chitri,Shoppers turn out for Black Friday deals as ho...,https://www.chicagotribune.com/2026/11/27/blac...
226,39,2026-11-27-16-53-46 +0000,bbc,Canada's top court largely upholds Quebec ban ...,https://www.bbc.co.uk/news/articles/cq3dgpy9le...
36,37,2026-11-28-11-00-00 +0000,startribune,"One year after the Rose Avenue ICE raid, quest...",https://www.startribune.com/one-year-after-the...
6,29,2026-11-28-13-06-31 +0000,nypost,Pregnant Cornell student claims sexist profess...,https://nypost.com/2026/11/28/us-news/pregnant...
230,29,2026-11-27-16-35-08 +0000,nypost,Real cause of death of Deb Fischer’s husband r...,https://nypost.com/2026/11/27/us-news/real-cau...
215,26,2026-11-27-17-41-21 +0000,cbc,Zelenskyy criticizes Trump for cozying up to P...,https://www.cbc.ca/news/world/zelensky-eu-ukra...
162,26,2026-11-27-20-47-01 +0000,wapo,Mexico condemns DEA tip line asking people to ...,https://www.washingtonpost.com/world/2026/11/2...
153,25,2026-11-27-21-21-19 +0000,nypost,NYC grocery prices jump nearly one-third since...,https://nypost.com/2026/11/27/business/nyc-gro...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
